In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

In [7]:
# 1. Memuat User-Item Matrix (Data Training 80%)
# Menggunakan steam_id sebagai index baris
train_matrix = pd.read_csv('user_item_matrix_train.csv', index_col='steam_id')

# PERBAIKAN BUG: Ubah tipe data header kolom dari String kembali ke Integer
train_matrix.columns = train_matrix.columns.astype(int)

# 2. Memuat Data Testing (Ground Truth 20%)
test_df = pd.read_csv('testing_set_groundtruth.csv')

print(f"Dimensi Matriks Pelatihan: {train_matrix.shape[0]} user x {train_matrix.shape[1]} game")
print(f"Total baris Data Pengujian: {test_df.shape[0]} interaksi tersembunyi")

Dimensi Matriks Pelatihan: 495 user x 5242 game
Total baris Data Pengujian: 6579 interaksi tersembunyi


In [9]:
# TAHAP 1: Menghitung Cosine Similarity
# Menghasilkan matriks simetris berukuran (User x User)
cosine_sim_array = cosine_similarity(train_matrix)

# Ubah ke bentuk DataFrame agar mudah dilacak menggunakan steam_id
cosine_sim_df = pd.DataFrame(
    cosine_sim_array, 
    index=train_matrix.index, 
    columns=train_matrix.index
)

# TAHAP 2: Menghitung Euclidean Distance & Mengubahnya menjadi Similarity
euclidean_dist_array = euclidean_distances(train_matrix)

# Transformasi jarak menjadi skor kemiripan (semakin tinggi semakin mirip)
euclidean_sim_array = 1 / (1 + euclidean_dist_array)

# Ubah ke bentuk DataFrame
euclidean_sim_df = pd.DataFrame(
    euclidean_sim_array, 
    index=train_matrix.index, 
    columns=train_matrix.index
)

print("Matriks Kemiripan Pengguna telah berhasil dibentuk!")
print(f"Dimensi Matriks Kemiripan: {cosine_sim_df.shape}")

print("\nContoh 5 baris pertama dari Cosine Similarity Matrix:")
display(cosine_sim_df.head())

Matriks Kemiripan Pengguna telah berhasil dibentuk!
Dimensi Matriks Kemiripan: (495, 495)

Contoh 5 baris pertama dari Cosine Similarity Matrix:


steam_id,76561197963033154,76561197963786815,76561197986095555,76561197996826561,76561197999326811,76561198001244067,76561198004857374,76561198005704816,76561198012145612,76561198016531576,...,76561199750485721,76561199750937214,76561199767213080,76561199774736783,76561199796959899,76561199801150324,76561199814528330,76561199837041222,76561199840096740,76561199840738495
steam_id,,,,,,,,,,,,,,,,,,,,,
76561197963033154,1.000000,0.999414,0.611997,0.114473,0.832837,0.663998,0.000006,0.993398,0.882267,0.078349,...,0.998819,0.999300,0.998624,0.998994,0.070583,0.999073,0.418461,0.000000e+00,0.822712,0.002517
76561197963786815,0.999414,1.000000,0.610948,0.114539,0.831805,0.664268,0.000000,0.993961,0.882777,0.078395,...,0.999154,0.999753,0.999166,0.999553,0.070624,0.999651,0.418703,0.000000e+00,0.823187,0.000000
76561197986095555,0.611997,0.610948,1.000000,0.069978,0.708356,0.431521,0.006656,0.607264,0.539846,0.060323,...,0.612295,0.611887,0.610657,0.610746,0.650805,0.610740,0.256060,4.264506e-04,0.785509,0.014383
76561197996826561,0.114473,0.114539,0.069978,1.000000,0.107707,0.076124,0.002998,0.113848,0.110106,0.017994,...,0.114443,0.114512,0.114444,0.114489,0.008089,0.114500,0.063155,3.752802e-01,0.094288,0.012188
76561197999326811,0.832837,0.831805,0.708356,0.107707,1.000000,0.576610,0.000068,0.845252,0.737523,0.223702,...,0.833299,0.832740,0.831239,0.831795,0.058746,0.831521,0.349768,7.134944e-07,0.684737,0.001419


In [11]:
def get_top_n_recommendations(target_user, sim_matrix, train_matrix, k=20, n=10):
    """
    Menghasilkan Top-N rekomendasi game untuk target_user.
    
    Parameter:
    - target_user: ID Steam pengguna yang ingin dicarikan rekomendasi
    - sim_matrix: Matriks kemiripan (Cosine atau Euclidean DataFrame)
    - train_matrix: Matriks data latih (User-Item Matrix)
    - k: Jumlah tetangga terdekat (Neighbor Size)
    - n: Jumlah rekomendasi game yang ingin ditampilkan (Top-N)
    """
    
    # 1. Ambil skor kemiripan untuk pengguna target terhadap semua pengguna lain
    user_similarities = sim_matrix.loc[target_user]
    
    # 2. Urutkan dari skor tertinggi, buang diri sendiri, ambil K tetangga terdekat
    # (Diri sendiri dibuang karena kemiripan dengan diri sendiri pasti skornya maksimal)
    nearest_neighbors = user_similarities.drop(target_user).sort_values(ascending=False).head(k)
    
    # 3. Ambil data riwayat bermain dari K tetangga yang terpilih
    neighbors_data = train_matrix.loc[nearest_neighbors.index]
    
    # 4. Hitung Skor Prediksi (Weighted Sum)
    # Rumus: Jumlahkan (Skor Kemiripan Tetangga * Bobot Playtime Tetangga pada game i)
    # Menggunakan dot product matriks agar proses komputasi instan
    predicted_scores = nearest_neighbors.values.dot(neighbors_data.values)
    
    # Jadikan bentuk Pandas Series agar mudah difilter menggunakan app_id
    predicted_scores_series = pd.Series(predicted_scores, index=train_matrix.columns)
    
    # 5. Filter Game yang Sudah Dimainkan
    # Rekomendasi hanya valid untuk game yang poinnya masih 0 di matriks target_user
    target_user_played = train_matrix.loc[target_user]
    games_not_played = target_user_played[target_user_played == 0].index
    
    # Saring prediksi hanya pada game yang belum pernah disentuh pengguna target
    valid_recommendations = predicted_scores_series.loc[games_not_played]
    
    # 6. Urutkan dari skor prediksi tertinggi dan ambil N terbaik
    top_n_recommendations = valid_recommendations.sort_values(ascending=False).head(n)
    
    # Mengembalikan daftar app_id dalam bentuk list
    return top_n_recommendations.index.tolist()

print("Fungsi get_top_n_recommendations berhasil dimuat!")

Fungsi get_top_n_recommendations berhasil dimuat!


In [12]:
# TAHAP 1: Persiapan Ground Truth
# Mengelompokkan data testing (yang disembunyikan) menjadi dictionary
# Format: {steam_id: set(app_id1, app_id2, ...)}
ground_truth_dict = test_df.groupby('steam_id')['app_id'].apply(set).to_dict()

# TAHAP 2: Definisi Variasi Parameter Uji
k_values = [5, 10, 20, 50]
n_values = [5, 10, 15, 20]

# List untuk menampung hasil akhir baris per baris
evaluation_results = []

print("Memulai proses evaluasi massal... Ini akan memakan waktu beberapa menit.\n")

# Looping Skenario Metrik
for metric_name, sim_matrix in [('Cosine', cosine_sim_df), ('Euclidean', euclidean_sim_df)]:
    # Looping variasi K
    for k in k_values:
        # Looping variasi N
        for n in n_values:
            precision_list = []
            recall_list = []
            
            # Evaluasi dihitung untuk setiap user yang ada di ground truth
            for target_user, true_items in ground_truth_dict.items():
                # Pencegahan error jika user tidak ada di matriks training
                if target_user not in train_matrix.index:
                    continue
                
                # Eksekusi mesin rekomendasi
                recommended_items = get_top_n_recommendations(
                    target_user=target_user, 
                    sim_matrix=sim_matrix, 
                    train_matrix=train_matrix, 
                    k=k, 
                    n=n
                )
                
                recommended_set = set(recommended_items)
                
                # Hitung Irisan (Berapa banyak tebakan yang benar/relevan)
                hits = len(recommended_set.intersection(true_items))
                
                # Hitung Rumus Evaluasi
                precision = hits / n if n > 0 else 0
                recall = hits / len(true_items) if len(true_items) > 0 else 0
                
                precision_list.append(precision)
                recall_list.append(recall)
            
            # Menghitung nilai rata-rata dari seluruh user untuk kombinasi saat ini
            avg_precision = np.mean(precision_list)
            avg_recall = np.mean(recall_list)
            
            # Menyimpan data ke dalam struktur tabel
            evaluation_results.append({
                'Metrik Jarak': metric_name,
                'K (Neighbors)': k,
                'N (Top-N)': n,
                'Precision@N': round(avg_precision, 4),
                'Recall@N': round(avg_recall, 4)
            })
            print(f"Selesai: {metric_name} | K = {k:2} | N = {n:2}")

# TAHAP 3: Konversi ke DataFrame dan Tampilkan Hasil
df_results = pd.DataFrame(evaluation_results)

print("\n" + "="*50)
print("=== TABEL HASIL EVALUASI KOMPARATIF (BAB IV) ===")
print("="*50)
display(df_results)

# Simpan ke CSV agar angkanya tidak hilang dan mudah disalin ke Microsoft Word
df_results.to_csv('hasil_evaluasi_skripsi.csv', index=False)
print("\n[SUKSES] Data telah diekspor ke 'hasil_evaluasi_skripsi.csv'")

Memulai proses evaluasi massal... Ini akan memakan waktu beberapa menit.

Selesai: Cosine | K =  5 | N =  5
Selesai: Cosine | K =  5 | N = 10
Selesai: Cosine | K =  5 | N = 15
Selesai: Cosine | K =  5 | N = 20
Selesai: Cosine | K = 10 | N =  5
Selesai: Cosine | K = 10 | N = 10
Selesai: Cosine | K = 10 | N = 15
Selesai: Cosine | K = 10 | N = 20
Selesai: Cosine | K = 20 | N =  5
Selesai: Cosine | K = 20 | N = 10
Selesai: Cosine | K = 20 | N = 15
Selesai: Cosine | K = 20 | N = 20
Selesai: Cosine | K = 50 | N =  5
Selesai: Cosine | K = 50 | N = 10
Selesai: Cosine | K = 50 | N = 15
Selesai: Cosine | K = 50 | N = 20
Selesai: Euclidean | K =  5 | N =  5
Selesai: Euclidean | K =  5 | N = 10
Selesai: Euclidean | K =  5 | N = 15
Selesai: Euclidean | K =  5 | N = 20
Selesai: Euclidean | K = 10 | N =  5
Selesai: Euclidean | K = 10 | N = 10
Selesai: Euclidean | K = 10 | N = 15
Selesai: Euclidean | K = 10 | N = 20
Selesai: Euclidean | K = 20 | N =  5
Selesai: Euclidean | K = 20 | N = 10
Selesai: Euc

,Metrik Jarak,K (Neighbors),N (Top-N),Precision@N,Recall@N
0,Cosine,5,5,0.0941,0.0594
1,Cosine,5,10,0.0741,0.0856
2,Cosine,5,15,0.0641,0.1028
3,Cosine,5,20,0.0568,0.1191
4,Cosine,10,5,0.1127,0.0701
5,Cosine,10,10,0.0840,0.0978
6,Cosine,10,15,0.0711,0.1201
7,Cosine,10,20,0.0647,0.1439
8,Cosine,20,5,0.1236,0.0746
9,Cosine,20,10,0.0935,0.1076



[SUKSES] Data telah diekspor ke 'hasil_evaluasi_skripsi.csv'
